# SEQ 2 SEQ

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.functional as F

import math
import warnings
import numpy
import pandas

import json
import re
from collections import Counter
from pathlib import Path


In [2]:
warnings.filterwarnings("ignore", category = FutureWarning)
torch.set_float32_matmul_precision("high")


## DEVICE SETUP

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
print(f"Device in use: {device}")


Device in use: cuda


## PARAMETER

In [4]:
vocab_input = 200
vocab_output = 200
d_model = 64
d_ff = d_model * 4
num_heads = 4
num_layers = 4
dropout = 0.1


## POS ENCODER

In [5]:
def positional_encoder(seq_length, d_model):
    
    pe = torch.zeros(seq_length, d_model)
    pos = torch.arange(0, seq_length, dtype = torch.float32).unsqueeze(1)
    div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000) / d_model))
    
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    
    return pe


## HELPER

In [6]:
def safe_tensor(x):
    
    return x if torch.is_tensor(x) else torch.tensor(x, dtype = torch.float32).device


## ENCODER SINGLE LAYER

In [7]:
class encoder_layer(nn.Module):
    
    def __init__(self, d_model = d_model, d_ff = d_ff, num_heads = num_heads, dropout = dropout):
        super(encoder_layer, self).__init__()
        
        # norm
        
        self.norm1 = nn.LayerNorm(d_model)
        
        # attention layer
        
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first = True)
        
        # norm
        
        self.norm2 = nn.LayerNorm(d_model)
        
        # FFN
        
        self.ffn = nn.Sequential(
            
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )
        
        # dropout
        
        self.dropout = nn.Dropout(dropout)
        
        # apply optimization
        
        self.apply(self.optim)
        
    def optim(self, modulue):
        
        for m in modulue.modules():
            
            if isinstance(m, nn.Linear):
                
                nn.init.normal_(m.weight, 0, 0.02)
                
                if m.bias is not None:
                    
                    nn.init.zeros_(m.bias)
                    
            if isinstance(m, nn.MultiheadAttention):
                
                nn.init.normal_(m.in_proj_weight, 0, 0.02)
                
                if m.in_proj_bias is not None:
                    
                    nn.init.zeros_(m.in_proj_bias)
                    
                nn.init.normal_(m.out_proj.weight, 0, 0.02)
                
                if m.out_proj.bias is not None:
                    
                    nn.init.zeros_(m.out_proj.bias)
                    
    def forward(self, x, mask = None):
        
        # norm1
        
        norm1 = self.norm1(x)
        
        # mha
        
        attn, _ = self.attn(norm1, norm1, norm1, key_padding_mask = mask)
        
        # residual
        
        x = x + self.dropout(attn)
        
        # norm2
        
        norm2 = self.norm2(x)
        
        # ffn
        
        ffn = self.ffn(norm2)
        
        # residual
        
        x = x + self.dropout(ffn)
        
        return x


## ENCODER

In [8]:
class encoder_stack(nn.Module):
    
    def __init__(self, d_model = d_model, vocab_input = vocab_input, num_layers = num_layers):
        super(encoder_stack, self).__init__()
        
        # Embedding layer
        
        self.embed = nn.Embedding(vocab_input, d_model)
        
        # d_model
        
        self.d_model = d_model
        
        # stack
        
        self.layers = nn.ModuleList([
            
            encoder_layer() for layer in range(num_layers)
        ])
        
        # norm
        
        self.norm = nn.LayerNorm(d_model)
        
        
    def forward(self, x, mask = None, Uncover = False):
        
        # get position
        
        seq_length = x.size(1)
        if Uncover: print(f'Seq length: {seq_length}')
        
        pos = positional_encoder(seq_length, self.d_model).to(x.device)
        
        # embed
        
        embed = self.embed(x) * math.sqrt(self.d_model)
        
        if Uncover: print(f"pos size: {pos.size()} \npos shape: {pos.shape}\nembed size: {embed.size()}\n embed shape: {embed.shape}")

        # formulate input
        
        x = embed + pos
        
        # now layer stack
        
        for layer in self.layers:
            
            x = layer(x, mask = mask)
            
        if Uncover: print(f'X shape after layer: {x.shape}\nX size: {x.size()}')
        
        # layer norm
        
        norm = self.norm(x)
        
        return norm


## INIT

In [9]:
Encoder = encoder_stack().to(device)
print(Encoder)

print("\n----------------------------------------------")

params = sum(p.numel() for p in Encoder.parameters())
print(f"No of params: {params}")


encoder_stack(
  (embed): Embedding(200, 64)
  (layers): ModuleList(
    (0-3): 4 x encoder_layer(
      (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
      )
      (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ffn): Sequential(
        (0): Linear(in_features=64, out_features=256, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=256, out_features=64, bias=True)
      )
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
)

----------------------------------------------
No of params: 212864


## DECODER SINGLE LAYER

In [10]:
class decoder_layer(nn.Module):
    
    def __init__(self, d_model = d_model, d_ff = d_ff, dropout = dropout, num_heads = num_heads):
        super(decoder_layer, self).__init__()
        
        # layer norm
        
        self.norm = nn.LayerNorm(d_model)
        
        # Attn
        
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first = True)
        
        # layer norm
        
        self.norm2 = nn.LayerNorm(d_model)
        
        # cross attn
        
        self.cross_Attn = nn.MultiheadAttention(d_model, num_heads, batch_first = True)
        
        # norm 3
        
        self.norm3 = nn.LayerNorm(d_model)
        
        # FFN
        
        self.ffn = nn.Sequential(
            
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )
        
        # dropout 
        
        self.dropout = nn.Dropout(dropout)
        
        # optimization
        
        self.apply(self.__init_weight__)
        
    def forward(self, x, enc_out, attn_mask = None, target_mask = None, enc_mask = None):
        
        # x -> norm
        
        norm = self.norm(x)
        
        # norm -> attn
        
        attn, _ = self.attn(
            norm,
            norm,
            norm,
            attn_mask = attn_mask,
            key_padding_mask = target_mask
        )
        
        # residual
        
        x = x + self.dropout(attn)
        
        # residual -> norm
        
        norm2 = self.norm2(x)
        
        # cross attn
        
        cross_attn, _ = self.cross_Attn(
            norm2,
            enc_out,
            enc_out,
            key_padding_mask = enc_mask
        )
        
        # cross_attn -> residual
        
        x = x + self.dropout(cross_attn)
        
        # norm 3 
        
        norm3 = self.norm3(x)
        
        # norm -> FFN
        
        ffn = self.ffn(norm3)
        
        # residual
        
        x = x + self.dropout(ffn)
        
        return x
        
    
    def __init_weight__(self, modulue):
        
        for m in modulue.modules():
            
            if isinstance(m, nn.Linear):
                
                nn.init.normal_(m.weight, 0, 0.2)
                
                if m.bias is not None:
                    
                    nn.init.zeros_(m.bias)
                    
            if isinstance(m, nn.MultiheadAttention):
                
                nn.init.normal_(m.in_proj_weight, 0, 0.2)
                
                if m.in_proj_bias is not None:
                    
                    nn.init.zeros_(m.in_proj_bias)
                
                nn.init.normal_(m.out_proj.weight, 0, 0.2)
                
                if m.out_proj.bias is not None:
                    
                    nn.init.zeros_(m.out_proj.bias)
                    
                

## DECODER

In [11]:
class decoder(nn.Module):
    
    def __init__(self, d_model = d_model, d_ff = d_ff, num_layers = num_layers, vocab_input = vocab_input):
        super(decoder, self).__init__()
        
        # D_model
        
        self.d_model = d_model
        
        # embed layer
        
        self.embed = nn.Embedding(vocab_input, d_model)
        
        # decoder layers
        
        self.layers = nn.ModuleList([
            
            decoder_layer() for _ in range(num_layers)
        ])
        
        # norm
        
        self.norm = nn.LayerNorm(d_model)
        
    def forward(self, x, enc_out, enc_mask = None, attn_mask = None, target_mask = None, inspect = False):
        
        # get embed
        
        embed = self.embed(x) * math.sqrt(self.d_model)
        
        if inspect: print(f'embed shape: {embed.shape}')
        
        # get pos encoded 
        
        seq_length = x.size(1)
        
        if inspect: print(f'Seq length: {seq_length}')
        
        pos = positional_encoder(seq_length, self.d_model).to(x.device)
        
        x = embed + pos
        
        if inspect: print(f'x size: {x.size()} | x shape: {x.shape}')
        
        for layer in self.layers:
            
            x = layer(
                x,
                enc_out,
                attn_mask = attn_mask,
                target_mask = target_mask,
                enc_mask = enc_mask
            )
            
        x = self.norm(x)
            
        return x


## INIT

In [12]:
Decoder = decoder().to(device)
print(Decoder)
print("--------------------------------------------------------------------")
params = sum(p.numel() for p in Decoder.parameters())
print(f"No. of params in Decoder: {params}")


decoder(
  (embed): Embedding(200, 64)
  (layers): ModuleList(
    (0-3): 4 x decoder_layer(
      (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
      )
      (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (cross_Attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
      )
      (norm3): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ffn): Sequential(
        (0): Linear(in_features=64, out_features=256, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=256, out_features=64, bias=True)
      )
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
)
--------------------------------------------------------------------
No. of params in Decoder: 2799

## SEQ2SEQ


In [13]:
class seq2seq(nn.Module):
    
    def __init__(self, d_model = d_model, vocab_input = vocab_input, vocab_output = vocab_output):
        super(seq2seq, self).__init__()

        self.encoder = encoder_stack(vocab_input = vocab_input).to(device)
        
        self.decoder = decoder(vocab_input = vocab_input).to(device)
        
        
        self.dec_projection = nn.Linear(d_model, vocab_output)
        
    def forward(self, enc_input, dec_input, enc_mask = None, dec_mask = None, target_mask = None):
        
        enc = self.encoder(enc_input, enc_mask)
        
        dec = self.decoder(
            dec_input,
            enc,
            enc_mask = enc_mask,
            attn_mask = dec_mask,
            target_mask = target_mask
        )
        
        dec_proj = self.dec_projection(dec)
        
        return dec_proj


## INIT

In [14]:
FULL_TRANSFORMER = seq2seq().to(device)
print(FULL_TRANSFORMER)
print("----------------------------------------------------")
params = sum(p.numel() for p in FULL_TRANSFORMER.parameters())
print(f'Params in complete transformer: {params}')


seq2seq(
  (encoder): encoder_stack(
    (embed): Embedding(200, 64)
    (layers): ModuleList(
      (0-3): 4 x encoder_layer(
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (ffn): Sequential(
          (0): Linear(in_features=64, out_features=256, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=256, out_features=64, bias=True)
        )
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  )
  (decoder): decoder(
    (embed): Embedding(200, 64)
    (layers): ModuleList(
      (0-3): 4 x decoder_layer(
        (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (attn): MultiheadAttention(
          (out_proj): NonDyna

## DATA LOADING

In [15]:
from torch.utils.data import DataLoader, Dataset

dataset_dir = next(
    (folder / 'cherry_seq2seq_dataset_v3' for folder in [Path.cwd(), *Path.cwd().parents] if (folder / 'cherry_seq2seq_dataset_v3').exists()),
    None
)

if dataset_dir is None:
    raise FileNotFoundError('Could not find cherry_seq2seq_dataset_v3. Open this notebook from the project folder.')

def load_jsonl(path):
    
    with open(path, 'r', encoding = 'utf-8') as file:
        return [json.loads(line) for line in file if line.strip()]

train_records = load_jsonl(dataset_dir / 'train.jsonl')
validation_records = load_jsonl(dataset_dir / 'validation.jsonl')
test_records = load_jsonl(dataset_dir / 'test.jsonl')

print(f'Train: {len(train_records):,} | Validation: {len(validation_records):,} | Test: {len(test_records):,}')
print(train_records[0])


Train: 40,000 | Validation: 5,000 | Test: 5,000
{'input': "We're facing this problem: marketing does not have predefined customer categories. Can you help me separate customers into naturally occurring groups? Return the result as JSON.", 'target': '{"domain":"machine_learning","task":"clustering","objective":"group customers into similar segments","inputs":["customer data"],"outputs":["customer segments"],"constraints":["return the result as JSON"],"tools":[],"requirements":["perform unsupervised clustering"]}'}


## PREPROCESS

In [16]:
special_tokens = ['<pad>', '<unk>', '<bos>', '<sep>', '<eos>']
token_pattern = re.compile(r"[a-z0-9]+|[^\w\s]", re.IGNORECASE)

def tokenize(text):
    
    return token_pattern.findall(text.lower())

token_counts = Counter()

for record in train_records:
    token_counts.update(tokenize(record['input']))
    token_counts.update(tokenize(record['target']))

token_to_id = {token: index for index, token in enumerate(special_tokens)}

for token, _ in token_counts.most_common():
    if token not in token_to_id:
        token_to_id[token] = len(token_to_id)

id_to_token = {index: token for token, index in token_to_id.items()}
pad_id = token_to_id['<pad>']
unk_id = token_to_id['<unk>']
bos_id = token_to_id['<bos>']
sep_id = token_to_id['<sep>']
eos_id = token_to_id['<eos>']

vocab_input = len(token_to_id)
vocab_output = vocab_input

print(f'Vocabulary size: {vocab_input:,}')


Vocabulary size: 467


## MASKING

In [17]:
max_input_length = 48
max_target_length = 128
batch_size = 32

def encode(text, max_length):
    
    tokens = tokenize(text)[:max_length]
    return [token_to_id.get(token, unk_id) for token in tokens]

class prompt_dataset(Dataset):
    
    def __init__(self, records):
        self.records = records
        
    def __len__(self):
        return len(self.records)
    
    def __getitem__(self, index):
        
        record = self.records[index]
        source = encode(record['input'], max_input_length)
        target = encode(record['target'], max_target_length)
        source_ids = [bos_id] + source + [eos_id]
        target_ids = [bos_id] + target + [eos_id]
        
        return {
            'source_ids': torch.tensor(source_ids, dtype = torch.long),
            'target_ids': torch.tensor(target_ids, dtype = torch.long)
        }

def collate_batch(batch):
    
    source_ids = nn.utils.rnn.pad_sequence(
        [item['source_ids'] for item in batch],
        batch_first = True,
        padding_value = pad_id
    )
    
    target_ids = nn.utils.rnn.pad_sequence(
        [item['target_ids'] for item in batch],
        batch_first = True,
        padding_value = pad_id
    )
    
    return {
        'source_ids': source_ids,
        'target_ids': target_ids,
        'source_mask': source_ids.eq(pad_id),
        'target_mask': target_ids.eq(pad_id)
    }

train_loader = DataLoader(prompt_dataset(train_records), batch_size = batch_size, shuffle = True, collate_fn = collate_batch)
validation_loader = DataLoader(prompt_dataset(validation_records), batch_size = batch_size, shuffle = False, collate_fn = collate_batch)
test_loader = DataLoader(prompt_dataset(test_records), batch_size = batch_size, shuffle = False, collate_fn = collate_batch)


## LOSS - OPTIMIZERS - SECHEDULER

In [18]:
Epoch = 20
learning_rate = 1e-3
warmup_epoch = 8
plateau_epochs = 4
decay_epoch = 8


In [19]:
FULL_TRANSFORMER = seq2seq(
    vocab_input = vocab_input,
    vocab_output = vocab_output
).to(device)

def causal_mask(sequence_length):
    
    return torch.triu(
        torch.ones(sequence_length, sequence_length, dtype = torch.bool, device = device),
        diagonal = 1
    )

model_loss = nn.CrossEntropyLoss(ignore_index = pad_id)
optimizer = optim.AdamW(
    FULL_TRANSFORMER.parameters(),
    lr = learning_rate
)

early_scheduler = optim.lr_scheduler.LinearLR(optimizer, total_iters = warmup_epoch)
plateau_scheduler = optim.lr_scheduler.ConstantLR(optimizer, factor = 1.0, total_iters = plateau_epochs)
later_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, decay_epoch, eta_min = 1e-5)
scheduler = optim.lr_scheduler.SequentialLR(
    optimizer,
    [early_scheduler, plateau_scheduler, later_scheduler],
    [warmup_epoch, warmup_epoch + plateau_epochs]
)


## TRAINING

In [20]:
train_history = []
validation_history = []
best_validation_loss = float('inf')

for epoch in range(Epoch):
    
    FULL_TRANSFORMER.train()
    train_loss = 0.0
    
    for batch in train_loader:
        
        source_ids = batch['source_ids'].to(device)
        target_ids = batch['target_ids'].to(device)
        source_mask = batch['source_mask'].to(device)
        decoder_input = target_ids[:, :-1]
        decoder_labels = target_ids[:, 1:]
        target_mask = decoder_input.eq(pad_id)
        attention_mask = causal_mask(decoder_input.size(1))
        
        logits = FULL_TRANSFORMER(
            source_ids,
            decoder_input,
            enc_mask = source_mask,
            dec_mask = attention_mask,
            target_mask = target_mask
        )
        loss = model_loss(
            logits.reshape(-1, vocab_output),
            decoder_labels.reshape(-1)
        )
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(FULL_TRANSFORMER.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    
    scheduler.step()
    train_loss /= len(train_loader)
    train_history.append(train_loss)
    
    FULL_TRANSFORMER.eval()
    validation_loss = 0.0
    
    with torch.no_grad():
        
        for batch in validation_loader:
            
            source_ids = batch['source_ids'].to(device)
            target_ids = batch['target_ids'].to(device)
            source_mask = batch['source_mask'].to(device)
            decoder_input = target_ids[:, :-1]
            decoder_labels = target_ids[:, 1:]
            target_mask = decoder_input.eq(pad_id)
            attention_mask = causal_mask(decoder_input.size(1))
            logits = FULL_TRANSFORMER(source_ids, decoder_input, enc_mask = source_mask, dec_mask = attention_mask, target_mask = target_mask)
            validation_loss += model_loss(logits.reshape(-1, vocab_output), decoder_labels.reshape(-1)).item()
    
    validation_loss /= len(validation_loader)
    validation_history.append(validation_loss)
    
    if validation_loss < best_validation_loss:
        
        best_validation_loss = validation_loss
        torch.save({'PROMPT ENHANCER': FULL_TRANSFORMER.state_dict(), 'token_to_id': token_to_id}, 'full_encoder_decoder.pt')
    
    print(f'Epoch {epoch + 1:03d} | Train Loss: {train_loss:.4f} | Validation Loss: {validation_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}')


Epoch 001 | Train Loss: 0.6633 | Validation Loss: 0.0157 | LR: 4.17e-04
Epoch 002 | Train Loss: 0.0118 | Validation Loss: 0.0016 | LR: 5.00e-04
Epoch 003 | Train Loss: 0.0036 | Validation Loss: 0.0010 | LR: 5.83e-04
Epoch 004 | Train Loss: 0.0024 | Validation Loss: 0.0003 | LR: 6.67e-04
Epoch 005 | Train Loss: 0.0014 | Validation Loss: 0.0002 | LR: 7.50e-04
Epoch 006 | Train Loss: 0.0012 | Validation Loss: 0.0002 | LR: 8.33e-04
Epoch 007 | Train Loss: 0.0012 | Validation Loss: 0.0001 | LR: 9.17e-04


c:\Users\kiosh\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\optim\lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


Epoch 008 | Train Loss: 0.0013 | Validation Loss: 0.0001 | LR: 1.00e-03
Epoch 009 | Train Loss: 0.0019 | Validation Loss: 0.0008 | LR: 1.00e-03
Epoch 010 | Train Loss: 0.0004 | Validation Loss: 0.0001 | LR: 1.00e-03
Epoch 011 | Train Loss: 0.0008 | Validation Loss: 0.0001 | LR: 1.00e-03
Epoch 012 | Train Loss: 0.0007 | Validation Loss: 0.0001 | LR: 1.00e-03
Epoch 013 | Train Loss: 0.0003 | Validation Loss: 0.0001 | LR: 9.62e-04
Epoch 014 | Train Loss: 0.0002 | Validation Loss: 0.0001 | LR: 8.55e-04
Epoch 015 | Train Loss: 0.0005 | Validation Loss: 0.0001 | LR: 6.94e-04
Epoch 016 | Train Loss: 0.0001 | Validation Loss: 0.0002 | LR: 5.05e-04
Epoch 017 | Train Loss: 0.0001 | Validation Loss: 0.0001 | LR: 3.16e-04
Epoch 018 | Train Loss: 0.0001 | Validation Loss: 0.0001 | LR: 1.55e-04
Epoch 019 | Train Loss: 0.0001 | Validation Loss: 0.0001 | LR: 4.77e-05
Epoch 020 | Train Loss: 0.0001 | Validation Loss: 0.0001 | LR: 1.00e-05


## TESTING


In [23]:
FULL_TRANSFORMER.eval()
test_loss = 0.0

with torch.no_grad():
    
    for batch in test_loader:
        
        source_ids = batch['source_ids'].to(device)
        target_ids = batch['target_ids'].to(device)
        source_mask = batch['source_mask'].to(device)
        decoder_input = target_ids[:, :-1]
        decoder_labels = target_ids[:, 1:]
        target_mask = decoder_input.eq(pad_id)
        attention_mask = causal_mask(decoder_input.size(1))
        logits = FULL_TRANSFORMER(source_ids, decoder_input, enc_mask = source_mask, dec_mask = attention_mask, target_mask = target_mask)
        test_loss += model_loss(logits.reshape(-1, vocab_output), decoder_labels.reshape(-1)).item()

test_loss /= len(test_loader)
print(f'Test Loss: {test_loss:.4f}')

def generate_refinement(prompt, max_new_tokens = max_target_length):
    
    source = encode(prompt, max_input_length)
    source_ids = torch.tensor([[bos_id] + source + [eos_id]], dtype = torch.long, device = device)
    source_mask = source_ids.eq(pad_id)
    generated = torch.tensor([[bos_id]], dtype = torch.long, device = device)
    
    with torch.no_grad():
        
        for _ in range(max_new_tokens):
            attention_mask = causal_mask(generated.size(1))
            target_mask = generated.eq(pad_id)
            logits = FULL_TRANSFORMER(source_ids, generated, enc_mask = source_mask, dec_mask = attention_mask, target_mask = target_mask)
            next_token = logits[:, -1].argmax(dim = -1)
            generated = torch.cat([generated, next_token.unsqueeze(1)], dim = 1)
            
            if next_token.item() == eos_id:
                break
    
    target_ids = generated[0].tolist()[1:]
    tokens = []
    
    for token_id in target_ids:
        
        if token_id == eos_id:
            break
        
        tokens.append(id_to_token.get(token_id, '<unk>'))
    
    return ' '.join(tokens).replace(' .', '.').replace(' ,', ',').replace(' :', ':')

example = test_records[100]
print('Poor / raw prompt:', example['input'])
print('Expected refinement:', example['target'])
print('Generated refinement:', generate_refinement(example['input']))


Test Loss: 0.0001
Poor / raw prompt: marketing does not have predefined customer categories. We have customer profiles. Rather than doing this manually, I need something that can group users according to behavioral similarity.
Expected refinement: {"domain":"machine_learning","task":"clustering","objective":"group customers into similar segments","inputs":["customer data"],"outputs":["customer segments"],"constraints":[],"tools":[],"requirements":["perform unsupervised clustering"]}
Generated refinement: { " domain ": " machine learning ", " task ": " clustering ", " objective ": " group customers into similar segments ", " inputs ": [ " customer data " ], " outputs ": [ " customer segments " ], " constraints ": [ ], " tools ": [ ], " requirements ": [ " perform unsupervised clustering " ] }
